In [ ]:
import os
os.chdir("/content")
if os.path.exists("structure_properties_fall_2026"):
    !git -C structure_properties_fall_2026 pull origin main
else:
    !git clone https://github.com/aldenbarnes/structure_properties_fall_2026.git
os.chdir("/content/structure_properties_fall_2026")
!ls Code
%pip install -q -r requirements.txt

In [2]:
import os, subprocess, sys
from google.colab import userdata

%pip install -q streamlit
os.chdir("/content/structure_properties_fall_2026")
os.environ["REALTYAPI_KEY"] = userdata.get("REALTYAPI_KEY")

log = open("streamlit.log", "w")
subprocess.Popen([sys.executable, "-m", "streamlit", "run", "Code/tool/rent_lookup_app.py",
                  "--server.headless", "true", "--server.port", "8501"],
                 stdout=log, stderr=log)

<Popen: returncode: None args: ['/usr/bin/python3', '-m', 'streamlit', 'run'...>

In [ ]:
import os, re, subprocess, time, urllib.request

def app_is_up():
    try:
        urllib.request.urlopen("http://localhost:8501", timeout=2)
        return True
    except Exception:
        return False

# Wait up to 60 seconds for the app started by the cell above.
for _ in range(60):
    if app_is_up():
        break
    time.sleep(1)

if not app_is_up():
    print("The app didn't start. End of streamlit.log:\n")
    print("".join(open("streamlit.log").readlines()[-20:]))
else:
    if not os.path.exists("cloudflared"):
        subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                        "-O", "cloudflared"], check=True)
        os.chmod("cloudflared", 0o755)
    subprocess.run(["pkill", "-f", "cloudflared tunnel"])   # close the tunnel from an earlier run, if any

    # The tunnel logs to cloudflared.log; only the link is printed. It is printed once the
    # tunnel is connected, so it opens right away.
    log = open("cloudflared.log", "w")
    tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                              stdout=log, stderr=subprocess.STDOUT)
    link = None
    for _ in range(60):
        time.sleep(1)
        text = open("cloudflared.log").read()
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
        link = link or (found.group(0) if found else None)
        if link and "Registered tunnel connection" in text:
            break

    if link:
        print(link)
        # Keep this cell running: the link works while it runs. Stop the cell to close the app.
        try:
            tunnel.wait()
        except KeyboardInterrupt:
            tunnel.terminate()
    else:
        tunnel.terminate()
        print("No link after 60 seconds. End of cloudflared.log:\n")
        print("".join(open("cloudflared.log").readlines()[-20:]))